# DataVisualisation-TP2025 — TP5

[Teacher assignment](https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP5/TP5.md)

Run the cells from top to bottom. The required answers appear at the end. The repository also contains `EXPLICATION_AR.md` with the complete Arabic explanation. This TP needs a reachable PostgreSQL database; the connection is entered privately below.

## Install the required libraries if needed

In [1]:
import importlib.util
import subprocess
import sys
packages = {'pandas': 'pandas', 'requests': 'requests', 'dlt': 'dlt[postgres]', 'dbt': 'dbt-postgres', 'prefect': 'prefect', 'psycopg2': 'psycopg2-binary'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


## Companion files for dlt and dbt

In [2]:
from pathlib import Path
# Recreate the companion files when opening only this notebook in Colab.
support_files = {'rest_api_pipeline.py': '"""Extract the teacher\'s sources and load PostgreSQL with dlt."""\nimport json\nimport os\nfrom pathlib import Path\nfrom urllib.parse import urlparse, unquote, parse_qs\n\nimport dlt\nimport pandas as pd\nimport requests\n\nROOT = Path(__file__).resolve().parent\nRATE_URL = "https://api.frankfurter.dev/v1/latest?base=USD&to=EUR,GBP,JPY"\nTRANSACTIONS_URL = "https://drive.google.com/uc?export=download&id=1oY9CIYmtY0nL78bVp3lxvCWVDKMuhFv7"\n\ndef configure_connection():\n    """One connection URL supplies both dlt and dbt; never print credentials."""\n    url = os.environ.get("DATABASE_URL")\n    if not url:\n        raise ValueError("Set DATABASE_URL to your PostgreSQL connection URL first (see README).")\n    parsed = urlparse(url)\n    if parsed.scheme not in {"postgres", "postgresql"}:\n        raise ValueError("DATABASE_URL must be a PostgreSQL URL.")\n    values = {"PGHOST": parsed.hostname or "localhost", "PGPORT": str(parsed.port or 5432),\n              "PGUSER": unquote(parsed.username or ""), "PGPASSWORD": unquote(parsed.password or ""),\n              "PGDATABASE": unquote(parsed.path.lstrip("/")),\n              "PGSSLMODE": parse_qs(parsed.query).get("sslmode", ["prefer"])[0]}\n    os.environ.update(values)\n    return url\n\ndef get_currency_rates():\n    response = requests.get(RATE_URL, timeout=60)\n    response.raise_for_status()\n    rates = response.json()\n    if rates["base"] != "USD" or rates["amount"] != 1 or any(rates["rates"][c] <= 0 for c in ["EUR", "GBP", "JPY"]):\n        raise ValueError("Expected positive EUR/GBP/JPY rates for one USD.")\n    (ROOT / "results").mkdir(exist_ok=True)\n    (ROOT / "results" / "exchange_rates.json").write_text(json.dumps(rates, indent=2), encoding="utf-8")\n    yield rates\n\ndef get_transaction_logs():\n    cached = ROOT / "data" / "transactions.csv"\n    return pd.read_csv(cached if cached.exists() else TRANSACTIONS_URL)\n\ndef load_pipeline():\n    url = configure_connection()\n    pipeline = dlt.pipeline(pipeline_name="currency_conversion", destination="postgres",\n                           dataset_name="raw", pipelines_dir=str(ROOT / ".dlt-state"))\n    # A nested rates dictionary becomes rates__eur, rates__gbp and rates__jpy.\n    pipeline.run(get_currency_rates(), table_name="currency_rates",\n                 write_disposition="replace", credentials=url)\n    transactions = get_transaction_logs()\n    pipeline.run(transactions, table_name="daily_transactions",\n                 write_disposition="append", credentials=url)\n    print(f"Loaded latest USD exchange rates and appended {len(transactions)} transactions to raw.")\n\nif __name__ == "__main__":\n    load_pipeline()\n', 'currency_flow.py': '"""The required dlt → dbt Prefect flow, once or scheduled daily."""\nimport os\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nos.environ.setdefault("PREFECT_HOME", str(ROOT / ".prefect"))\nos.environ.setdefault("PREFECT_SERVER_ANALYTICS_ENABLED", "false")\nos.environ.setdefault("DLT_TELEMETRY", "false")\n\nfrom dbt.cli.main import dbtRunner\nfrom prefect import flow, task\nfrom prefect.schedules import Cron\nfrom rest_api_pipeline import configure_connection, load_pipeline\n\n@task\ndef run_dbt_models():\n    project = ROOT / "currency_conversion"\n    result = dbtRunner().invoke(["run", "--project-dir", str(project), "--profiles-dir", str(project)])\n    if not result.success:\n        raise RuntimeError("dbt transformation failed; check the dbt log.") from result.exception\n\n@flow(name="Currency Exchange Rate ETL")\ndef currency_exchange_rate_etl():\n    configure_connection()\n    load_pipeline()\n    run_dbt_models()\n\nif __name__ == "__main__":\n    import sys\n    if "--serve" in sys.argv:\n        currency_exchange_rate_etl.serve(\n            name="daily-data-deployment", schedule=Cron("0 1 * * *", timezone="Africa/Lagos"),\n            description="Load exchange rates and transactions, then run dbt daily at 1 AM.",\n            tags=["daily-pipeline"],\n        )\n    else:\n        currency_exchange_rate_etl()\n', 'currency_conversion/dbt_project.yml': "name: currency_conversion\nversion: '1.0.0'\nconfig-version: 2\nprofile: currency_conversion\nmodel-paths: [models]\nmodels:\n  currency_conversion:\n    example:\n      +materialized: view\n", 'currency_conversion/profiles.yml': 'currency_conversion:\n  target: dev\n  outputs:\n    dev:\n      type: postgres\n      host: "{{ env_var(\'PGHOST\') }}"\n      port: "{{ env_var(\'PGPORT\') | int }}"\n      user: "{{ env_var(\'PGUSER\') }}"\n      password: "{{ env_var(\'PGPASSWORD\') }}"\n      dbname: "{{ env_var(\'PGDATABASE\') }}"\n      sslmode: "{{ env_var(\'PGSSLMODE\', \'prefer\') }}"\n      schema: public\n      threads: 1\n', 'currency_conversion/models/example/schema.yml': 'version: 2\nsources:\n  - name: raw_data_dlt\n    schema: raw\n    tables:\n      - name: currency_rates\n      - name: daily_transactions\n', 'currency_conversion/models/example/final_transaction_usd.sql': "with latest_rates as (\n    select date as rate_date, rates__eur as eur_rate,\n           rates__gbp as gbp_rate, rates__jpy as jpy_rate\n    from {{ source('raw_data_dlt', 'currency_rates') }}\n    order by date desc\n    limit 1\n)\nselect t.transaction_id, t.customer_id, t.transaction_date,\n       t.currency, t.amount as original_amount, r.rate_date,\n       case t.currency\n           when 'USD' then t.amount\n           when 'EUR' then t.amount / r.eur_rate\n           when 'GBP' then t.amount / r.gbp_rate\n           when 'JPY' then t.amount / r.jpy_rate\n           else null\n       end as amount_usd\nfrom {{ source('raw_data_dlt', 'daily_transactions') }} as t\ncross join latest_rates as r\nwhere t.amount is not null\n", 'data/transactions.csv': 'transaction_id,amount,currency,customer_id,transaction_date\nT-5001,100.00,EUR,CUST-901,2025-10-30\nT-5002,50.50,GBP,CUST-902,2025-10-30\nT-5003,15000.00,JPY,CUST-901,2025-10-30\nT-5004,25.00,EUR,CUST-903,2025-10-30\nT-5005,100.00,USD,CUST-902,2025-10-30\nT-5006,75.99,USD,CUST-904,2025-10-30\nT-5007,345.50,GBP,CUST-905,2025-10-30\nT-5008,1250.00,EUR,CUST-904,2025-10-30\nT-5009,200.00,USD,CUST-901,2025-10-30\nT-5010,8500.00,JPY,CUST-906,2025-10-30\nT-5011,15.20,EUR,CUST-905,2025-10-30\nT-5012,1000.00,JPY,CUST-907,2025-10-30\nT-5013,12.99,USD,CUST-903,2025-10-30'}
for name, content in support_files.items():
    path = Path(name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")


## 1. Imports and PostgreSQL connection

In [3]:
import json
import os
from pathlib import Path
from getpass import getpass

import pandas as pd
import psycopg2

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not os.environ.get("DATABASE_URL"):
    os.environ["DATABASE_URL"] = getpass("PostgreSQL connection URL (hidden): ")
from currency_flow import currency_exchange_rate_etl

## 2. Extract and load with dlt, then transform with dbt inside Prefect

In [4]:
currency_exchange_rate_etl()

Loaded latest USD exchange rates and appended 13 transactions to raw.
15:14:15  Running with dbt=1.12.5
15:14:16  Registered adapter: postgres=1.11.0
15:14:18  Unable to do partial parsing because profile has changed
15:14:21  Found 1 model, 2 sources, 477 macros
15:14:21  
15:14:21  Concurrency: 1 threads (target='dev')
15:14:21  
15:14:21  1 of 1 START sql view model public.final_transaction_usd ....................... [RUN]
15:14:21  1 of 1 OK created sql view model public.final_transaction_usd .................. [CREATE VIEW in 0.15s]
15:14:21  
15:14:21  Finished running 1 view model in 0 hours 0 minutes and 0.53 seconds (0.53s).
15:14:21  
15:14:21  Completed successfully
15:14:21  
15:14:21  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


## 3. Query the resulting USD view, without printing connection credentials

In [5]:
with psycopg2.connect(os.environ["DATABASE_URL"]) as connection:
    with connection.cursor() as cursor:
        cursor.execute("select * from public.final_transaction_usd order by transaction_id")
        final_transactions = pd.DataFrame(cursor.fetchall(), columns=[c.name for c in cursor.description])
print(final_transactions.to_string(index=False))
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
final_transactions.to_csv(OUT / "final_transaction_usd.csv", index=False)
rates = json.loads((OUT / "exchange_rates.json").read_text(encoding="utf-8"))
print("API rate date:", rates["date"], "Base:", rates["base"], "Rates:", rates["rates"])

transaction_id customer_id transaction_date currency  original_amount  rate_date  amount_usd
        T-5001    CUST-901       2025-10-30      EUR           100.00 2026-10-12  111.810548
        T-5002    CUST-902       2025-10-30      GBP            50.50 2026-10-12   66.838727
        T-5003    CUST-901       2025-10-30      JPY         15000.00 2026-10-12   94.577554
        T-5004    CUST-903       2025-10-30      EUR            25.00 2026-10-12   27.952637
        T-5005    CUST-902       2025-10-30      USD           100.00 2026-10-12  100.000000
        T-5006    CUST-904       2025-10-30      USD            75.99 2026-10-12   75.990000
        T-5007    CUST-905       2025-10-30      GBP           345.50 2026-10-12  457.282774
        T-5008    CUST-904       2025-10-30      EUR          1250.00 2026-10-12 1397.631853
        T-5009    CUST-901       2025-10-30      USD           200.00 2026-10-12  200.000000
        T-5010    CUST-906       2025-10-30      JPY          8500.00 

## 4. Explain the required pipeline and its conversion direction

In [6]:
report = f"""# TP5 — ETL result

The flow ran dlt extraction/loading before dbt transformation on PostgreSQL.
The teacher's CSV contains 13 transaction records. This database view currently
contains **{len(final_transactions)}** rows. Rates came from the real Frankfurter
API, base USD, dated **{rates['date']}**; the saved JSON records the exact rates.
Latest business-day rates may precede the execution date.

`raw.currency_rates` is replaced every run; `raw.daily_transactions` is appended
every run, as the assignment specifies. Replaying the same CSV adds the same
transactions again; scheduling this unchanged historical file is not an
idempotent production design. No unrequested merge/deduplication is introduced.
dbt builds `public.final_transaction_usd` as a **view** over those tables.
An API rate is foreign-currency units per one USD, so amount_usd = amount / rate.
USD amounts stay unchanged. Unknown currencies yield NULL rather than a fake rate.
This uses the latest rates for all transactions, including the historical 2025
records; it does not claim these are historical transaction-date conversions.

`python currency_flow.py` executes once. `python currency_flow.py --serve`
serves the required daily 01:00 schedule in Africa/Lagos; the process and
configured Prefect server must stay running for scheduled jobs to execute.
The local verification executed once; it did not leave a daily service running.
PostgreSQL credentials are supplied privately through DATABASE_URL, not committed.
The notebook also needs reachable PostgreSQL; in Colab, localhost refers to Colab.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP5/TP5.md
API: https://api.frankfurter.dev/v1/latest?base=USD&to=EUR,GBP,JPY
Transactions: https://drive.google.com/uc?export=download&id=1oY9CIYmtY0nL78bVp3lxvCWVDKMuhFv7
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Saved the USD results and pipeline explanation.")

Saved the USD results and pipeline explanation.


## Required written answers

In [7]:
from IPython.display import Markdown, display
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))

# TP5 — ETL result

The flow ran dlt extraction/loading before dbt transformation on PostgreSQL.
The teacher's CSV contains 13 transaction records. This database view currently
contains **13** rows. Rates came from the real Frankfurter
API, base USD, dated **2026-10-12**; the saved JSON records the exact rates.
Latest business-day rates may precede the execution date.

`raw.currency_rates` is replaced every run; `raw.daily_transactions` is appended
every run, as the assignment specifies. Replaying the same CSV adds the same
transactions again; scheduling this unchanged historical file is not an
idempotent production design. No unrequested merge/deduplication is introduced.
dbt builds `public.final_transaction_usd` as a **view** over those tables.
An API rate is foreign-currency units per one USD, so amount_usd = amount / rate.
USD amounts stay unchanged. Unknown currencies yield NULL rather than a fake rate.
This uses the latest rates for all transactions, including the historical 2025
records; it does not claim these are historical transaction-date conversions.

`python currency_flow.py` executes once. `python currency_flow.py --serve`
serves the required daily 01:00 schedule in Africa/Lagos; the process and
configured Prefect server must stay running for scheduled jobs to execute.
The local verification executed once; it did not leave a daily service running.
PostgreSQL credentials are supplied privately through DATABASE_URL, not committed.
The notebook also needs reachable PostgreSQL; in Colab, localhost refers to Colab.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP5/TP5.md
API: https://api.frankfurter.dev/v1/latest?base=USD&to=EUR,GBP,JPY
Transactions: https://drive.google.com/uc?export=download&id=1oY9CIYmtY0nL78bVp3lxvCWVDKMuhFv7
